# 📡 Phase 2 — Task 8: Fading Uygulanmış Dataset Üretimi

Bu notebook, orijinal RML2016.10a (AWGN) dataseti üzerine **Rayleigh** ve **Rician** fading kanallarını uygulayarak yeni dataset'ler üretir.

**Üretilecek dosyalar:**

| Dosya | Kanal Modeli | Açıklama |
|-------|-------------|----------|
| `RML2016.10a_rayleigh.pkl` | Rayleigh | NLOS ortam, derin sönümleme |
| `RML2016.10a_rician_K3.pkl` | Rician (K=3) | LOS bileşeni orta düzey |
| `RML2016.10a_rician_K10.pkl` | Rician (K=10) | LOS bileşeni baskın |

**Adımlar:**
1. Ortam kurulumu ve orijinal dataset yükleme
2. Rayleigh faded dataset üretimi
3. Rician faded dataset üretimi (K=3, K=10)
4. İstatistiksel doğrulama
5. IQ görselleştirme (orijinal vs faded karşılaştırması)
6. Dataset'leri Google Drive'a kaydetme

## 1. Ortam Kurulumu

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import time

print('✅ Kütüphaneler yüklendi.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/erigami-sl/AMR-UnderDifferentNoises-DL.git'
PROJECT_DIR = '/content/AMR-UnderDifferentNoises-DL'

if not os.path.exists(PROJECT_DIR):
    !git clone -b dev {REPO_URL}
else:
    # Mevcut repoyu güncelle
    !cd {PROJECT_DIR} && git pull origin dev
    print(f'Proje güncellendi: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f'✅ Çalışma dizini: {os.getcwd()}')

## 2. Orijinal Dataset Yükleme

Orijinal RML2016.10a dict dosyasını ham haliyle yüklüyoruz.  
`generate_faded_dataset()` fonksiyonu doğrudan bu dict üzerinde çalışır.

In [ ]:
DRIVE_DATASET = '/content/drive/MyDrive/AMR-Project/RML2016.10a_dict.pkl'
LOCAL_DATASET = os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_dict.pkl')

if os.path.exists(DRIVE_DATASET):
    DATASET_PATH = DRIVE_DATASET
elif os.path.exists(LOCAL_DATASET):
    DATASET_PATH = LOCAL_DATASET
else:
    raise FileNotFoundError(
        f'Dataset bulunamadı!\n'
        f'  Drive: {DRIVE_DATASET}\n'
        f'  Lokal: {LOCAL_DATASET}\n'
        f'Lütfen RML2016.10a_dict.pkl dosyasını yukarıdaki konumlardan birine yerleştirin.'
    )

Xd = pickle.load(open(DATASET_PATH, 'rb'), encoding='iso-8859-1')

# Temel bilgiler
mods = sorted(list(set([k[0] for k in Xd.keys()])))
snrs = sorted(list(set([k[1] for k in Xd.keys()])))

print(f'📁 Dataset: {DATASET_PATH}')
print(f'📊 Anahtar sayısı: {len(Xd.keys())} (beklenen: 220)')
print(f'📐 Örnek shape: {list(Xd.values())[0].shape} (beklenen: (1000, 2, 128))')
print(f'📡 Modülasyonlar ({len(mods)}): {mods}')
print(f'📶 SNR aralığı: {snrs[0]} → {snrs[-1]} dB ({len(snrs)} seviye)')
print(f'📦 Toplam örnek: {sum(v.shape[0] for v in Xd.values()):,}')

## 3. Faded Dataset Üretimi

Her kanal modeli için `generate_faded_dataset()` çağrılır.  
Tekrarlanabilirlik için sabit `seed` kullanılır (Task 7b'den gelen iyileştirme).

In [ ]:
from src.utils.channels import generate_faded_dataset
print('✅ channels.py import edildi.')

In [ ]:
# ── Üretim parametreleri ──
MASTER_SEED = 2016  # Projenin standart seed'i (config.py ile tutarlı)

CHANNEL_CONFIGS = {
    'rayleigh': {
        'channel_type': 'rayleigh',
        'K': None,
        'seed': MASTER_SEED,
        'filename': 'RML2016.10a_rayleigh.pkl',
        'label': 'Rayleigh Fading (NLOS)'
    },
    'rician_K3': {
        'channel_type': 'rician',
        'K': 3.0,
        'seed': MASTER_SEED + 1000,
        'filename': 'RML2016.10a_rician_K3.pkl',
        'label': 'Rician Fading (K=3, Moderate LOS)'
    },
    'rician_K10': {
        'channel_type': 'rician',
        'K': 10.0,
        'seed': MASTER_SEED + 2000,
        'filename': 'RML2016.10a_rician_K10.pkl',
        'label': 'Rician Fading (K=10, Strong LOS)'
    }
}

print('📋 Üretim planı:')
for name, cfg in CHANNEL_CONFIGS.items():
    K_str = f'K={cfg["K"]}' if cfg['K'] else 'N/A'
    print(f'  • {cfg["label"]:40s} → {cfg["filename"]}  (seed={cfg["seed"]}, {K_str})')

In [ ]:
# ── Dataset üretimi ──
faded_datasets = {}

for name, cfg in CHANNEL_CONFIGS.items():
    print(f'\n🔄 Üretiliyor: {cfg["label"]}...')
    
    kwargs = {
        'channel_type': cfg['channel_type'],
        'seed': cfg['seed']
    }
    if cfg['K'] is not None:
        kwargs['K'] = cfg['K']
    
    t0 = time.time()
    faded_datasets[name] = generate_faded_dataset(Xd, **kwargs)
    elapsed = time.time() - t0
    
    # Doğrulama
    n_keys = len(faded_datasets[name])
    n_samples = sum(v.shape[0] for v in faded_datasets[name].values())
    sample_shape = list(faded_datasets[name].values())[0].shape
    has_nan = any(np.isnan(v).any() for v in faded_datasets[name].values())
    
    print(f'   ✅ Tamamlandı: {elapsed:.2f}s')
    print(f'   📊 Anahtar: {n_keys}, Toplam örnek: {n_samples:,}, Shape: {sample_shape}')
    print(f'   🔍 NaN kontrolü: {"❌ NaN VAR!" if has_nan else "✅ Temiz"}')

print(f'\n🎉 Tüm dataset\'ler başarıyla üretildi!')

## 4. İstatistiksel Doğrulama

Orijinal (AWGN) ve fading uygulanmış sinyallerin güç istatistiklerini karşılaştırır.  
Rayleigh fading'de `E[|h|²]=1` olduğu için ortalama güç yaklaşık korunmalıdır.

In [ ]:
# ── SNR bazında güç karşılaştırması ──
test_mods = ['BPSK', 'QPSK', 'QAM16', '8PSK', 'WBFM']
test_snrs = [-10, 0, 10, 18]

print(f'{"Mod":>8} | {"SNR":>4} | {"AWGN Güç":>10} | {"Rayleigh":>10} | {"Ric. K=3":>10} | {"Ric. K=10":>10} | {"Ray/AWGN":>8} | {"R3/AWGN":>8} | {"R10/AWGN":>8}')
print('-' * 110)

for mod in test_mods:
    for snr in test_snrs:
        key = (mod, snr)
        if key not in Xd:
            continue
        
        p_awgn = np.mean(Xd[key]**2)
        p_ray  = np.mean(faded_datasets['rayleigh'][key]**2)
        p_r3   = np.mean(faded_datasets['rician_K3'][key]**2)
        p_r10  = np.mean(faded_datasets['rician_K10'][key]**2)
        
        print(f'{mod:>8} | {snr:>4} | {p_awgn:>10.6f} | {p_ray:>10.6f} | {p_r3:>10.6f} | {p_r10:>10.6f} | {p_ray/p_awgn:>8.4f} | {p_r3/p_awgn:>8.4f} | {p_r10/p_awgn:>8.4f}')

print('\n💡 Rayleigh güç oranı ~1.0, Rician K arttıkça orana daha yakın (LOS baskınlaşır).')

## 5. IQ Görselleştirme — Orijinal vs Faded Karşılaştırması

Seçili modülasyon ve SNR için zaman serisi ve constellation diyagramları.

In [ ]:
# ── Görselleştirme: Zaman serisi + Constellation ──
VIS_MODS = ['QPSK', 'QAM16', '8PSK']
VIS_SNR  = 10
SAMPLE_IDX = 0

for mod_name in VIS_MODS:
    key = (mod_name, VIS_SNR)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    
    datasets = [
        ('Orijinal (AWGN)', Xd[key]),
        ('Rayleigh',        faded_datasets['rayleigh'][key]),
        ('Rician K=3',      faded_datasets['rician_K3'][key]),
        ('Rician K=10',     faded_datasets['rician_K10'][key])
    ]
    
    for col, (title, X) in enumerate(datasets):
        # Üst satır: Zaman serisi
        axes[0, col].plot(X[SAMPLE_IDX, 0, :], label='I', alpha=0.8, linewidth=0.8, color='#2196F3')
        axes[0, col].plot(X[SAMPLE_IDX, 1, :], label='Q', alpha=0.8, linewidth=0.8, color='#FF5722')
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')
        axes[0, col].legend(fontsize=8)
        axes[0, col].grid(True, alpha=0.3)
        axes[0, col].set_ylabel('Genlik')
        axes[0, col].set_xlabel('Örnek')
        
        # Alt satır: IQ constellation (ilk 50 örnek üst üste)
        n_show = 50
        I_all = X[:n_show, 0, :].flatten()
        Q_all = X[:n_show, 1, :].flatten()
        axes[1, col].scatter(I_all, Q_all, s=1, alpha=0.15, color='#4CAF50')
        axes[1, col].set_xlabel('I')
        axes[1, col].set_ylabel('Q')
        axes[1, col].set_aspect('equal')
        axes[1, col].grid(True, alpha=0.3)
        axes[1, col].set_title(f'{title} — Constellation')
    
    fig.suptitle(f'{mod_name} — SNR={VIS_SNR}dB — Kanal Karşılaştırması', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# ── SNR etkisi görselleştirmesi: Tek modülasyon, farklı SNR seviyeleri ──
VIS_MOD = 'QPSK'
VIS_SNRS = [-10, 0, 10, 18]

fig, axes = plt.subplots(len(VIS_SNRS), 4, figsize=(20, 4 * len(VIS_SNRS)))

channel_names = ['AWGN (Orijinal)', 'Rayleigh', 'Rician K=3', 'Rician K=10']
channel_data  = [Xd, faded_datasets['rayleigh'], faded_datasets['rician_K3'], faded_datasets['rician_K10']]

for row, snr_val in enumerate(VIS_SNRS):
    for col, (ch_name, ch_data) in enumerate(zip(channel_names, channel_data)):
        key = (VIS_MOD, snr_val)
        X = ch_data[key]
        
        n_show = 30
        I_all = X[:n_show, 0, :].flatten()
        Q_all = X[:n_show, 1, :].flatten()
        axes[row, col].scatter(I_all, Q_all, s=1, alpha=0.15, color='#3F51B5')
        axes[row, col].set_aspect('equal')
        axes[row, col].grid(True, alpha=0.3)
        
        if row == 0:
            axes[row, col].set_title(ch_name, fontsize=12, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(f'SNR={snr_val}dB', fontsize=11, fontweight='bold')

fig.suptitle(f'{VIS_MOD} — SNR vs Kanal Etkisi (Constellation)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 6. Dataset'leri Kaydetme

Üretilen faded dataset'ler aynı pickle dict formatında kaydedilir.  
Anahtar yapısı orijinal ile aynıdır: `(modulation, snr) → ndarray(1000, 2, 128)`.

In [ ]:
# ── Kayıt dizinleri ──
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AMR-Project'
LOCAL_SAVE_DIR = os.path.join(PROJECT_DIR, 'data')

# Drive varsa oraya, yoksa lokale kaydet
SAVE_DIR = DRIVE_SAVE_DIR if os.path.exists('/content/drive/MyDrive') else LOCAL_SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'💾 Kayıt dizini: {SAVE_DIR}')

for name, cfg in CHANNEL_CONFIGS.items():
    filepath = os.path.join(SAVE_DIR, cfg['filename'])
    
    with open(filepath, 'wb') as f:
        pickle.dump(faded_datasets[name], f, protocol=pickle.HIGHEST_PROTOCOL)
    
    size_mb = os.path.getsize(filepath) / 1024 / 1024
    print(f'  ✅ {cfg["filename"]:40s} — {size_mb:.1f} MB')

print(f'\n🎉 Tüm dataset\'ler kaydedildi!')

In [ ]:
# ── Kayıt doğrulaması: Dosyaları geri yükleyip kontrol et ──
print('🔍 Kayıt doğrulaması...\n')

for name, cfg in CHANNEL_CONFIGS.items():
    filepath = os.path.join(SAVE_DIR, cfg['filename'])
    
    Xd_loaded = pickle.load(open(filepath, 'rb'))
    
    # Anahtar sayısı
    assert len(Xd_loaded) == 220, f'{name}: Anahtar sayısı {len(Xd_loaded)} != 220'
    
    # Shape kontrolü
    for key in Xd_loaded:
        assert Xd_loaded[key].shape == (1000, 2, 128), f'{name}/{key}: Shape {Xd_loaded[key].shape}'
    
    # NaN kontrolü
    assert not any(np.isnan(v).any() for v in Xd_loaded.values()), f'{name}: NaN bulundu!'
    
    # Orijinal ile aynı olmadığını kontrol et (fading uygulanmış olmalı)
    test_key = ('QPSK', 10)
    assert not np.array_equal(Xd_loaded[test_key], Xd[test_key]), f'{name}: Orijinal ile aynı!'
    
    print(f'  ✅ {cfg["label"]:45s} — 220 anahtar, tüm shape\'ler doğru, NaN yok, fading uygulanmış')

print('\n✅ Tüm dosyalar doğrulandı. Dataset\'ler eğitime hazır!')

## 7. Özet

### Üretilen dataset dosyaları

In [ ]:
print('=' * 70)
print('📋 DATASET ÜRETİM ÖZETİ')
print('=' * 70)
print(f'\nKaynak dataset: RML2016.10a_dict.pkl (AWGN)')
print(f'Master seed:    {MASTER_SEED}')
print(f'Kayıt dizini:   {SAVE_DIR}')
print(f'\n{"Dosya":40s} | {"Kanal":25s} | {"Boyut":>8}')
print('-' * 80)

for name, cfg in CHANNEL_CONFIGS.items():
    filepath = os.path.join(SAVE_DIR, cfg['filename'])
    size_mb = os.path.getsize(filepath) / 1024 / 1024
    print(f'{cfg["filename"]:40s} | {cfg["label"]:25s} | {size_mb:>6.1f} MB')

print(f'\nHer dataset:')
print(f'  • 220 anahtar (11 modülasyon × 20 SNR seviyesi)')
print(f'  • 220,000 toplam örnek')
print(f'  • Format: (1000, 2, 128) — orijinal ile aynı')
print(f'\n✅ Sonraki adım: Bu dataset\'ler ile model eğitimi (Task 9)')
print(f'   → notebooks/05_train_rayleigh.ipynb')
print(f'   → notebooks/06_train_rician.ipynb')

---
## ✅ Task 8 Tamamlandı

Fading uygulanmış dataset'ler başarıyla üretildi ve Google Drive'a kaydedildi.

**Sonraki adım:** Bu dataset'ler üzerinde MCLDNN ve PET-CGDNN eğitimleri (Task 9).